# Notebook 03: Gold-Layer - Business-Aggregationen und KPIs

## Übersicht
Dieses Notebook implementiert die **Gold-Layer-Komponente** der Medallion-Architektur. Es erstellt geschäftsfertige, aggregierte Daten für Analytics und Reporting aus den validierten Silver-Layer-Daten.

## Medallion Architecture - Gold Layer
Der Gold-Layer dient als **Analytics-Ready Zone** mit folgenden Eigenschaften:
- ✅ Zeitbasierte Aggregationen (stündlich, täglich)
- ✅ Business-KPIs und Metriken
- ✅ Statistische Analysen (Durchschnitt, Min, Max, Standardabweichung)
- ✅ Optimiert für schnelle Abfragen
- ✅ Reduzierte Datenmenge für BI-Tools

## Business Value

### Warum Gold Layer?
1. **Performance**: Voraggregierte Daten = schnellere Dashboards
2. **Konsistenz**: Standardisierte Business-Metriken
3. **Einfachheit**: Einfache Struktur für BI-Analysten
4. **Effizienz**: Reduzierte Datenmenge (von Millionen auf Tausende)

### Aggregations-Level
- **Stündlich**: Detaillierte Trend-Analyse
- **Täglich**: Langzeit-Monitoring
- **Pro Gerät**: Device-spezifische KPIs

### Business Metriken
- Durchschnittswerte (Temperatur, Luftfeuchtigkeit)
- Min/Max (Extremwerte erkennen)
- Standardabweichung (Variabilität messen)
- Anzahl Messungen (Datenqualität)
- Zeitfenster (erste/letzte Messung)

## Datenfluss
```
Silver Table (Validierte Daten)
    ↓
Filter: Nur is_valid = True
    ↓
Zeitbasierte Gruppierung (Stunde)
    ↓
Statistische Aggregationen
    ↓
Business-Metriken berechnen
    ↓
Gold Table (Analytics-Ready)
```

## Optimierungen
- **Partitionierung**: Nach aggregation_date
- **Z-Ordering**: device_id + hour_timestamp
- **Reduzierte Zeilen**: ~99% weniger als Silver
- **Schnellere Queries**: Millisekunden statt Sekunden

---

In [0]:
# BIBLIOTHEKEN UND ABHÄNGIGKEITEN


# PySpark Core
from pyspark.sql.functions import (
    col, count, avg, min as spark_min, max as spark_max, stddev,
    first, last, date_trunc, to_date, current_timestamp, round as spark_round,
    when, lit
)

# Python Standard
from datetime import datetime

print("✓ Alle Bibliotheken erfolgreich importiert")
print(f"Notebook gestartet: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ Alle Bibliotheken erfolgreich importiert
Notebook gestartet: 2026-03-01 01:56:05


In [0]:
# KONFIGURATION


# Tabellen-Konfiguration
SILVER_TABLE = "databricks_workspace.default.iot_data_silver"
GOLD_TABLE = "databricks_workspace.default.iot_data_gold"

# Aggregations-Einstellungen
AGGREGATION_LEVEL = "hourly"  # Stündliche Aggregationen
DECIMAL_PLACES = 2            # Dezimalstellen für Metriken

print("✓ Konfiguration geladen")
print(f"\n📊 GOLD-LAYER-KONFIGURATION:")
print("="*60)
print(f"  Quell-Tabelle       : {SILVER_TABLE}")
print(f"  Ziel-Tabelle        : {GOLD_TABLE}")
print(f"  Aggregations-Level  : {AGGREGATION_LEVEL}")
print(f"  Dezimalstellen      : {DECIMAL_PLACES}")
print("="*60)

print(f"\n🎯 BUSINESS-METRIKEN:")
print("  Zeitbasiert:")
print("    - Stündliche Aggregationen")
print("    - Pro Gerät gruppiert")
print("  Statistische Kennzahlen:")
print("    - Durchschnitt (avg)")
print("    - Minimum (min)")
print("    - Maximum (max)")
print("    - Standardabweichung (stddev)")
print("  Datenqualität:")
print("    - Anzahl Messungen")
print("    - Zeitfenster (erste/letzte Messung)")
print("="*60)

✓ Konfiguration geladen

📊 GOLD-LAYER-KONFIGURATION:
  Quell-Tabelle       : databricks_workspace.default.iot_data_silver
  Ziel-Tabelle        : databricks_workspace.default.iot_data_gold
  Aggregations-Level  : hourly
  Dezimalstellen      : 2

🎯 BUSINESS-METRIKEN:
  Zeitbasiert:
    - Stündliche Aggregationen
    - Pro Gerät gruppiert
  Statistische Kennzahlen:
    - Durchschnitt (avg)
    - Minimum (min)
    - Maximum (max)
    - Standardabweichung (stddev)
  Datenqualität:
    - Anzahl Messungen
    - Zeitfenster (erste/letzte Messung)


In [0]:
# SILVER-DATEN LESEN (NUR VALIDIERTE)


print("📖 Lese validierte Daten aus Silver-Layer...\n")

# Lese nur valide Datensätze aus Silver
silver_clean = (
    spark.read.format("delta").table(SILVER_TABLE)
    .filter(col("is_valid") == True)  # Nur validierte Daten
)

total_valid_records = silver_clean.count()

print(f"✓ Silver-Daten erfolgreich geladen")
print(f"  - Tabelle: {SILVER_TABLE}")
print(f"  - Valide Datensätze: {total_valid_records:,}")
print(f"  - Filter: is_valid = True")

if total_valid_records == 0:
    print("\n⚠️ WARNUNG: Keine validen Datensätze gefunden!")
    print("   Bitte Notebook 03 (Silver Layer) ausführen.")
else:
    print(f"\n✓ Bereit für Aggregation")
    
# Zeige Schema
print("\n📋 Silver-Daten-Schema (Auswahl):")
silver_clean.select(
    "device_id", "temperature", "humidity", 
    "sensor_timestamp_dt", "ingestion_date"
).printSchema()

📖 Lese validierte Daten aus Silver-Layer...

✓ Silver-Daten erfolgreich geladen
  - Tabelle: databricks_workspace.default.iot_data_silver
  - Valide Datensätze: 15
  - Filter: is_valid = True

✓ Bereit für Aggregation

📋 Silver-Daten-Schema (Auswahl):
root
 |-- device_id: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- sensor_timestamp_dt: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = true)



In [0]:
# STÜNDLICHE AGGREGATIONEN ERSTELLEN


print("🔄 Erstelle stündliche Aggregationen...\n")

# Erstelle stündliche Aggregationen pro Gerät
gold_hourly = (
    silver_clean
    .withColumn("hour_timestamp", date_trunc("hour", col("sensor_timestamp_dt")))
    .groupBy("device_id", "hour_timestamp")
    .agg(
        # Anzahl Messungen
        count("*").alias("reading_count"),
        
        # Temperatur-Statistiken
        spark_round(avg("temperature"), DECIMAL_PLACES).alias("avg_temperature"),
        spark_round(spark_min("temperature"), DECIMAL_PLACES).alias("min_temperature"),
        spark_round(spark_max("temperature"), DECIMAL_PLACES).alias("max_temperature"),
        spark_round(stddev("temperature"), DECIMAL_PLACES).alias("stddev_temperature"),
        
        # Luftfeuchtigkeits-Statistiken
        spark_round(avg("humidity"), DECIMAL_PLACES).alias("avg_humidity"),
        spark_round(spark_min("humidity"), DECIMAL_PLACES).alias("min_humidity"),
        spark_round(spark_max("humidity"), DECIMAL_PLACES).alias("max_humidity"),
        spark_round(stddev("humidity"), DECIMAL_PLACES).alias("stddev_humidity"),
        
        # Zeitfenster
        first("sensor_timestamp_dt").alias("first_reading_time"),
        last("sensor_timestamp_dt").alias("last_reading_time")
    )
    .withColumn("aggregation_date", to_date(col("hour_timestamp")))
    .withColumn("created_at", current_timestamp())
)

print("✓ Stündliche Aggregationen erstellt")
print(f"\n📊 AGGREGATIONS-DETAILS:")
print("="*60)
print(f"  Gruppierung         : device_id + Stunde")
print(f"  Temperatur-Metriken : avg, min, max, stddev")
print(f"  Feuchtigkeits-Metriken : avg, min, max, stddev")
print(f"  Zusätzliche Metriken : reading_count, Zeitfenster")
print(f"  Partitionierung     : aggregation_date")
print("="*60)

# Zeige Schema
print("\n📋 Gold-Layer-Schema:")
gold_hourly.printSchema()

🔄 Erstelle stündliche Aggregationen...

✓ Stündliche Aggregationen erstellt

📊 AGGREGATIONS-DETAILS:
  Gruppierung         : device_id + Stunde
  Temperatur-Metriken : avg, min, max, stddev
  Feuchtigkeits-Metriken : avg, min, max, stddev
  Zusätzliche Metriken : reading_count, Zeitfenster
  Partitionierung     : aggregation_date

📋 Gold-Layer-Schema:
root
 |-- device_id: string (nullable = true)
 |-- hour_timestamp: timestamp (nullable = true)
 |-- reading_count: long (nullable = false)
 |-- avg_temperature: double (nullable = true)
 |-- min_temperature: double (nullable = true)
 |-- max_temperature: double (nullable = true)
 |-- stddev_temperature: double (nullable = true)
 |-- avg_humidity: double (nullable = true)
 |-- min_humidity: double (nullable = true)
 |-- max_humidity: double (nullable = true)
 |-- stddev_humidity: double (nullable = true)
 |-- first_reading_time: timestamp (nullable = true)
 |-- last_reading_time: timestamp (nullable = true)
 |-- aggregation_date: date (nul

In [0]:
# AGGREGIERTE DATEN VORSCHAU


print("🔍 Vorschau der aggregierten Daten...\n")

total_aggregations = gold_hourly.count()

print(f"📊 AGGREGATIONS-STATISTIKEN:")
print("="*60)
print(f"  Anzahl Aggregationen : {total_aggregations:,}")
print(f"  Eingangsdatensätze   : {total_valid_records:,}")
if total_valid_records > 0:
    reduction = ((total_valid_records - total_aggregations) / total_valid_records) * 100
    print(f"  Datenreduktion       : {reduction:.2f}%")
    print(f"  Verdichtungsfaktor   : {total_valid_records / total_aggregations if total_aggregations > 0 else 0:.1f}x")
print("="*60)

print("\n📄 BEISPIEL-AGGREGATIONEN (Top 10):")
sample_data = (
    gold_hourly
    .orderBy(col("hour_timestamp").desc())
    .limit(10)
    .select(
        "device_id",
        "hour_timestamp",
        "reading_count",
        "avg_temperature",
        "avg_humidity",
        "min_temperature",
        "max_temperature"
    )
)

display(sample_data)

🔍 Vorschau der aggregierten Daten...

📊 AGGREGATIONS-STATISTIKEN:
  Anzahl Aggregationen : 15
  Eingangsdatensätze   : 15
  Datenreduktion       : 0.00%
  Verdichtungsfaktor   : 1.0x

📄 BEISPIEL-AGGREGATIONEN (Top 10):


device_id,hour_timestamp,reading_count,avg_temperature,avg_humidity,min_temperature,max_temperature
sensor_002,2026-03-01T01:00:00.000Z,1,28.71,56.76,28.71,28.71
sensor_004,2026-03-01T01:00:00.000Z,1,15.86,33.21,15.86,15.86
sensor_003,2026-03-01T01:00:00.000Z,1,28.9,61.48,28.9,28.9
sensor_005,2026-03-01T01:00:00.000Z,1,21.34,71.71,21.34,21.34
sensor_001,2026-03-01T01:00:00.000Z,1,24.31,50.64,24.31,24.31
sensor_002,2026-03-01T00:00:00.000Z,1,22.46,60.02,22.46,22.46
sensor_001,2026-03-01T00:00:00.000Z,1,17.91,42.51,17.91,17.91
sensor_003,2026-03-01T00:00:00.000Z,1,17.64,65.3,17.64,17.64
sensor_005,2026-03-01T00:00:00.000Z,1,22.66,46.1,22.66,22.66
sensor_004,2026-03-01T00:00:00.000Z,1,17.95,67.66,17.95,17.95


In [0]:
# BUSINESS-KPIS BERECHNEN


print("📈 Berechne zusätzliche Business-KPIs...\n")

# Erweitere Gold-Layer mit Business-KPIs
gold_with_kpis = (
    gold_hourly
    .withColumn(
        "temp_range",
        spark_round(col("max_temperature") - col("min_temperature"), DECIMAL_PLACES)
    )
    .withColumn(
        "humidity_range",
        spark_round(col("max_humidity") - col("min_humidity"), DECIMAL_PLACES)
    )
    .withColumn(
        "temp_stability_score",
        when(col("stddev_temperature") < 1.0, lit("STABIL"))
        .when(col("stddev_temperature") < 3.0, lit("NORMAL"))
        .otherwise(lit("VARIABEL"))
    )
    .withColumn(
        "data_quality_score",
        when(col("reading_count") >= 10, lit("HOCH"))
        .when(col("reading_count") >= 5, lit("MITTEL"))
        .otherwise(lit("NIEDRIG"))
    )
)

print("✓ Business-KPIs erfolgreich hinzugefügt")
print(f"\n💼 NEUE BUSINESS-METRIKEN:")
print("="*60)
print("  1. temp_range            : Temperaturspanne (max - min)")
print("  2. humidity_range        : Feuchtigkeitsspanne (max - min)")
print("  3. temp_stability_score  : Temperaturstabilität")
print("     - STABIL   : stddev < 1.0°C")
print("     - NORMAL   : stddev < 3.0°C")
print("     - VARIABEL : stddev >= 3.0°C")
print("  4. data_quality_score    : Datenqualität basiert auf Anzahl")
print("     - HOCH    : >= 10 Messungen")
print("     - MITTEL  : 5-9 Messungen")
print("     - NIEDRIG : < 5 Messungen")
print("="*60)

# Zeige Beispiel mit KPIs
print("\n📊 BEISPIEL MIT BUSINESS-KPIS:")
display(
    gold_with_kpis
    .select(
        "device_id",
        "hour_timestamp",
        "avg_temperature",
        "temp_range",
        "temp_stability_score",
        "reading_count",
        "data_quality_score"
    )
    .limit(5)
)

📈 Berechne zusätzliche Business-KPIs...

✓ Business-KPIs erfolgreich hinzugefügt

💼 NEUE BUSINESS-METRIKEN:
  1. temp_range            : Temperaturspanne (max - min)
  2. humidity_range        : Feuchtigkeitsspanne (max - min)
  3. temp_stability_score  : Temperaturstabilität
     - STABIL   : stddev < 1.0°C
     - NORMAL   : stddev < 3.0°C
     - VARIABEL : stddev >= 3.0°C
  4. data_quality_score    : Datenqualität basiert auf Anzahl
     - HOCH    : >= 10 Messungen
     - MITTEL  : 5-9 Messungen
     - NIEDRIG : < 5 Messungen

📊 BEISPIEL MIT BUSINESS-KPIS:


device_id,hour_timestamp,avg_temperature,temp_range,temp_stability_score,reading_count,data_quality_score
sensor_001,2026-02-28T23:00:00.000Z,16.76,0.0,VARIABEL,1,NIEDRIG
sensor_001,2026-03-01T00:00:00.000Z,17.91,0.0,VARIABEL,1,NIEDRIG
sensor_001,2026-03-01T01:00:00.000Z,24.31,0.0,VARIABEL,1,NIEDRIG
sensor_002,2026-02-28T23:00:00.000Z,15.83,0.0,VARIABEL,1,NIEDRIG
sensor_002,2026-03-01T00:00:00.000Z,22.46,0.0,VARIABEL,1,NIEDRIG


In [0]:
# GOLD DELTA TABLE SCHREIBEN


print("💾 Schreibe aggregierte Daten zu Gold-Table...\n")

# Schreibe zu Gold Delta Table
gold_with_kpis.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("aggregation_date") \
    .saveAsTable(GOLD_TABLE)

print("\n" + "="*60)
print("✓ GOLD-LAYER ERFOLGREICH ERSTELLT")
print("="*60)
print(f"  ✓ Tabelle: {GOLD_TABLE}")
print(f"  ✓ Format: Delta Lake")
print(f"  ✓ Aggregationen: {total_aggregations:,}")
print(f"  ✓ Partitionierung: aggregation_date")
print(f"  ✓ Business-KPIs: 4 zusätzliche Metriken")
print("="*60)

💾 Schreibe aggregierte Daten zu Gold-Table...


✓ GOLD-LAYER ERFOLGREICH ERSTELLT
  ✓ Tabelle: databricks_workspace.default.iot_data_gold
  ✓ Format: Delta Lake
  ✓ Aggregationen: 15
  ✓ Partitionierung: aggregation_date
  ✓ Business-KPIs: 4 zusätzliche Metriken


In [0]:
# GOLD-TABELLE VERIFIZIEREN


print("⚙️ Verifiziere Gold-Layer...\n")

# 1. Lese Gold-Tabelle
gold_table = spark.read.format("delta").table(GOLD_TABLE)
total_gold_records = gold_table.count()

print("📊 GOLD-LAYER-STATISTIKEN:")
print("="*60)
print(f"  Gesamtanzahl Aggregationen : {total_gold_records:,}")
print(f"  Anzahl Spalten             : {len(gold_table.columns)}")

# 2. Device-Statistiken
device_count = gold_table.select("device_id").distinct().count()
print(f"  Anzahl Geräte              : {device_count}")

# 3. Zeitraum
if total_gold_records > 0:
    time_stats = gold_table.agg(
        spark_min("hour_timestamp").alias("earliest"),
        spark_max("hour_timestamp").alias("latest")
    ).collect()[0]
    print(f"\n⏰ ZEITRAUM:")
    print(f"  Früheste Aggregation : {time_stats['earliest']}")
    print(f"  Späteste Aggregation : {time_stats['latest']}")

# 4. Qualitätsverteilung
print(f"\n📊 DATENQUALITÄTS-VERTEILUNG:")
quality_dist = gold_table.groupBy("data_quality_score").count().orderBy("count", ascending=False)
quality_dist.show(truncate=False)

# 5. Temperatur-Stabilitätsverteilung
print(f"\n🌡️ TEMPERATUR-STABILITÄTS-VERTEILUNG:")
stability_dist = gold_table.groupBy("temp_stability_score").count().orderBy("count", ascending=False)
stability_dist.show(truncate=False)

print("="*60)

⚙️ Verifiziere Gold-Layer...

📊 GOLD-LAYER-STATISTIKEN:
  Gesamtanzahl Aggregationen : 15
  Anzahl Spalten             : 19
  Anzahl Geräte              : 5

⏰ ZEITRAUM:
  Früheste Aggregation : 2026-02-28 23:00:00
  Späteste Aggregation : 2026-03-01 01:00:00

📊 DATENQUALITÄTS-VERTEILUNG:
+------------------+-----+
|data_quality_score|count|
+------------------+-----+
|NIEDRIG           |15   |
+------------------+-----+


🌡️ TEMPERATUR-STABILITÄTS-VERTEILUNG:
+--------------------+-----+
|temp_stability_score|count|
+--------------------+-----+
|VARIABEL            |15   |
+--------------------+-----+



In [0]:
# TABELLEN-DETAILS


print("📋 GOLD-TABELLEN-DETAILS:\n")

# DESCRIBE DETAIL für Delta-Tabelle
table_details = spark.sql(f"DESCRIBE DETAIL {GOLD_TABLE}").collect()[0]

print("="*60)
print("DELTA LAKE EIGENSCHAFTEN:")
print("="*60)
print(f"  Tabellen-ID          : {table_details['id']}")
print(f"  Format               : {table_details['format']}")
print(f"  Speicherort          : {table_details['location']}")
print(f"  Anzahl Dateien       : {table_details['numFiles']}")
print(f"  Größe (MB)           : {table_details['sizeInBytes'] / (1024*1024):.2f}")
print(f"  Partitions-Spalten   : {table_details['partitionColumns']}")
print("="*60)

# Beispiel-Abfrage für BI
print("\n💡 BEISPIEL-BI-ABFRAGE:")
print("-" * 60)
print("-- Durchschnittliche Temperatur pro Gerät (letzte 24h)")
print("SELECT")
print("  device_id,")
print("  AVG(avg_temperature) as daily_avg_temp,")
print("  AVG(avg_humidity) as daily_avg_humidity,")
print("  SUM(reading_count) as total_readings")
print(f"FROM {GOLD_TABLE}")
print("WHERE hour_timestamp >= current_timestamp() - INTERVAL 24 HOURS")
print("GROUP BY device_id")
print("ORDER BY daily_avg_temp DESC;")
print("-" * 60)

📋 GOLD-TABELLEN-DETAILS:

DELTA LAKE EIGENSCHAFTEN:
  Tabellen-ID          : c1006b7e-31a7-4d19-9c1c-4018c46dd1b2
  Format               : delta
  Speicherort          : abfss://unity-catalog-storage@dbstoragelnlclc344kuts.dfs.core.windows.net/7405615313946546/__unitystorage/catalogs/22fe63c2-d2a8-42f7-9d14-c568097c1073/tables/60df2e41-6428-40b8-8f28-c5f7fdaa095a
  Anzahl Dateien       : 2
  Größe (MB)           : 0.01
  Partitions-Spalten   : ['aggregation_date']

💡 BEISPIEL-BI-ABFRAGE:
------------------------------------------------------------
-- Durchschnittliche Temperatur pro Gerät (letzte 24h)
SELECT
  device_id,
  AVG(avg_temperature) as daily_avg_temp,
  AVG(avg_humidity) as daily_avg_humidity,
  SUM(reading_count) as total_readings
FROM databricks_workspace.default.iot_data_gold
WHERE hour_timestamp >= current_timestamp() - INTERVAL 24 HOURS
GROUP BY device_id
ORDER BY daily_avg_temp DESC;
------------------------------------------------------------


In [0]:
# ZUSAMMENFASSUNG UND NÄCHSTE SCHRITTE


print("\n" + "🎉 "*30)
print("\n📊 GOLD-LAYER-ZUSAMMENFASSUNG")
print("\n" + "="*60)

# Finale Statistiken
final_gold = spark.read.format("delta").table(GOLD_TABLE)
total_aggs = final_gold.count()

print(f"\n1️⃣ DATEN-TRANSFORMATION:")
print(f"   ✓ Quelle                : {SILVER_TABLE}")
print(f"   ✓ Ziel                  : {GOLD_TABLE}")
print(f"   ✓ Input (validiert)     : {total_valid_records:,} Datensätze")
print(f"   ✓ Output (aggregiert)   : {total_aggs:,} Datensätze")

if total_valid_records > 0 and total_aggs > 0:
    reduction_pct = ((total_valid_records - total_aggs) / total_valid_records) * 100
    compression = total_valid_records / total_aggs
    print(f"   ✓ Datenreduktion        : {reduction_pct:.2f}%")
    print(f"   ✓ Verdichtungsfaktor    : {compression:.1f}x")

print(f"\n2️⃣ AGGREGATIONS-LEVEL:")
print(f"   ✓ Zeiteinheit           : Stündlich")
print(f"   ✓ Gruppierung           : Pro Gerät (device_id)")
print(f"   ✓ Statistiken           : avg, min, max, stddev")
print(f"   ✓ Dezimalstellen        : {DECIMAL_PLACES}")

print(f"\n3️⃣ BUSINESS-KPIS:")
print(f"   ✓ Temperaturspanne      : max - min")
print(f"   ✓ Feuchtigkeitsspanne   : max - min")
print(f"   ✓ Stabilitäts-Score     : Basiert auf Standardabweichung")
print(f"   ✓ Qualitäts-Score       : Basiert auf Anzahl Messungen")

print(f"\n4️⃣ OPTIMIERUNGEN:")
print(f"   ✓ Format                : Delta Lake (ACID)")
print(f"   ✓ Partitionierung       : aggregation_date")
print(f"   ✓ Empfohlenes Z-Ordering: device_id + hour_timestamp")
print(f"   ✓ BI-optimiert          : Ja")

print(f"\n5️⃣ VERWENDUNGSZWECKE:")
print(f"   📊 Dashboards           : Echtzeit-Monitoring")
print(f"   📈 Trend-Analyse        : Zeitreihen-Visualisierung")
print(f"   🔔 Alerting             : Schwellenwert-Überwachung")
print(f"   📑 Reporting            : Periodische Berichte")
print(f"   🤖 ML-Features          : Feature Engineering")

print(f"\n6️⃣ NÄCHSTE SCHRITTE:")
print(f"   → Notebook 05: Performance-Optimization ausführen")
print(f"   → Z-Ordering auf Gold-Tabelle anwenden")
print(f"   → OPTIMIZE für schnellere Queries")
print(f"   → Notebook 06: Dashboards und Visualisierungen")

print("\n" + "="*60)
print("✓ NOTEBOOK 04 ERFOLGREICH AUSGEFÜHRT")
print("="*60)
print("\n" + "🎉 "*30)

print("\n🎯 GOLD-LAYER BEREIT FÜR:")
print("   ✅ Business Intelligence Tools (Power BI, Tableau)")
print("   ✅ SQL-Abfragen und Analytics")
print("   ✅ Real-time Dashboards")
print("   ✅ Machine Learning Pipelines")
print("   ✅ Automatisierte Berichte")


🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 

📊 GOLD-LAYER-ZUSAMMENFASSUNG


1️⃣ DATEN-TRANSFORMATION:
   ✓ Quelle                : databricks_workspace.default.iot_data_silver
   ✓ Ziel                  : databricks_workspace.default.iot_data_gold
   ✓ Input (validiert)     : 15 Datensätze
   ✓ Output (aggregiert)   : 15 Datensätze
   ✓ Datenreduktion        : 0.00%
   ✓ Verdichtungsfaktor    : 1.0x

2️⃣ AGGREGATIONS-LEVEL:
   ✓ Zeiteinheit           : Stündlich
   ✓ Gruppierung           : Pro Gerät (device_id)
   ✓ Statistiken           : avg, min, max, stddev
   ✓ Dezimalstellen        : 2

3️⃣ BUSINESS-KPIS:
   ✓ Temperaturspanne      : max - min
   ✓ Feuchtigkeitsspanne   : max - min
   ✓ Stabilitäts-Score     : Basiert auf Standardabweichung
   ✓ Qualitäts-Score       : Basiert auf Anzahl Messungen

4️⃣ OPTIMIERUNGEN:
   ✓ Format                : Delta Lake (ACID)
   ✓ Partitionierung       : aggregation_date
   ✓ Empfohlenes Z-Ordering: device_id + hour_timesta